<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.2/blob/main/%E3%80%87ILT%EF%BC%8BBlox_plot%EF%BC%88AddD2O_CPMG%EF%BC%89_2605.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# COMPLETE INTEGRATED CODE
# AddD2O_CPMG decay curves -> ILT
# + 3D stacked ILT figure
# + 1x3 box plot figure
# + ilt_summary_ridge_improved.csv
# + ZIP export
# ============================================================

!pip -q install numpy pandas matplotlib scipy openpyxl

import os
import re
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import lsq_linear
from scipy.signal import find_peaks
from mpl_toolkits.mplot3d.art3d import PolyCollection
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

warnings.filterwarnings("ignore")

# ============================================================
# 1. Settings
# ============================================================
CSV_PATH = "/content/AddD2O_CPMG_T2_Relative_Intensity_smoothed_44copolymers.csv"

OUTDIR = "/content/AddD2O_CPMG_ILT_summary_figures_zip"
os.makedirs(OUTDIR, exist_ok=True)

# ---------- ILT ----------
N_T2 = 240
RIDGE_ALPHA = 0.02
NORMALIZE_BY_FIRST_POINT = True

USE_AUTO_T2_RANGE = True
MANUAL_T2_MIN_MS = 0.1
MANUAL_T2_MAX_MS = 5000.0

# ---------- Component fraction ----------
SHORT_T2_MAX_MS = 1.0
MID_T2_MAX_MS = 30.0

# ---------- Peak detection ----------
PEAK_HEIGHT_REL = 0.05
PEAK_DISTANCE_POINTS = 5

# ---------- Classification thresholds ----------
BROAD_WIDTH_THRESHOLD = 0.70
MULTI_PEAK_MIN = 3

# ---------- Group order ----------
CROSSLINKER_ORDER = ["DICL", "TRCL", "TECL"]

# ---------- Colors ----------
CURTAIN_COLORS = {
    "DICL":  "#1B5E20",
    "TRCL":  "#2E7D32",
    "TECL":  "#66BB6A",
    "OTHER": "#999999",
}

GROUP_COLORS = {
    "DICL": "#1B5E20",
    "TRCL": "#2E7D32",
    "TECL": "#66BB6A",
    "OTHER": "#999999",
}

# ============================================================
# 2. Font and figure controls
# ============================================================
GLOBAL_FONT_SCALE = 1.35

BASE_AXIS_LABEL_FONTSIZE = 16
BASE_TICK_FONTSIZE_X = 12
BASE_TICK_FONTSIZE_Y = 12
BASE_TICK_FONTSIZE_Z = 12
BASE_GROUP_LABEL_FONTSIZE = 13
BASE_TITLE_FONTSIZE = 18

AXIS_LABEL_FONTSIZE = int(BASE_AXIS_LABEL_FONTSIZE * GLOBAL_FONT_SCALE)
TICK_FONTSIZE_X = int(BASE_TICK_FONTSIZE_X * GLOBAL_FONT_SCALE)
TICK_FONTSIZE_Y = int(BASE_TICK_FONTSIZE_Y * GLOBAL_FONT_SCALE)
TICK_FONTSIZE_Z = int(BASE_TICK_FONTSIZE_Z * GLOBAL_FONT_SCALE)
GROUP_LABEL_FONTSIZE = int(BASE_GROUP_LABEL_FONTSIZE * GLOBAL_FONT_SCALE)
TITLE_FONTSIZE = int(BASE_TITLE_FONTSIZE * GLOBAL_FONT_SCALE)

X_LABELPAD = 14
Y_LABELPAD = 18
Z_LABELPAD = 12

BASE_BOX_TITLE_SIZE = 16
BASE_BOX_LABEL_SIZE = 13
BASE_BOX_TICK_SIZE = 11

BOX_TITLE_SIZE = int(BASE_BOX_TITLE_SIZE * GLOBAL_FONT_SCALE)
BOX_LABEL_SIZE = int(BASE_BOX_LABEL_SIZE * GLOBAL_FONT_SCALE)
BOX_TICK_SIZE = int(BASE_BOX_TICK_SIZE * GLOBAL_FONT_SCALE)

# ---------- 3D ILT layout ----------
CURTAIN_FIG_W = 15
CURTAIN_FIG_H = 12
CURTAIN_ELEV = 28
CURTAIN_AZIM = -64

FACE_ALPHA = 0.90
POLY_EDGE_COLOR = "black"
POLY_EDGE_LW = 0.7

SHOW_PROFILE_LINES = True
PROFILE_LINE_COLOR = "black"
PROFILE_LINE_LW = 0.8
PROFILE_LINE_ALPHA = 0.95

SHOW_GROUP_SEPARATORS = True
GROUP_SEPARATOR_COLOR = "gray"
GROUP_SEPARATOR_LS = "--"
GROUP_SEPARATOR_LW = 0.9

SHOW_GROUP_TEXT = True
XMAX_LOG10 = 5.0
GROUP_LABEL_X = 4.55
ZMAX_CURTAIN = 0.2

XTICKS_3 = [0, 2, 4]
XTICKLABELS_3 = [r"$10^0$", r"$10^2$", r"$10^4$"]

# ---------- Box plot layout ----------
BOX_FIG_W = 16
BOX_FIG_H = 5.2
BOX_DPI = 300

POINT_ALPHA = 0.90
POINT_SIZE = 42
JITTER_WIDTH = 0.12

BOX_WIDTH = 0.55
BOX_FACE_ALPHA = 0.22
BOX_LINEWIDTH = 1.4
MEDIAN_LINEWIDTH = 2.0

# ============================================================
# 3. Helper functions
# ============================================================
def parse_float_ms(x):
    s = str(x).strip().replace("ms", "").replace("MS", "")
    return float(s)


def make_unique_names_with_index_for_duplicates(names):
    names = [str(n) for n in names]
    total_counts = {}
    for n in names:
        total_counts[n] = total_counts.get(n, 0) + 1

    seen_counts = {}
    out = []
    for n in names:
        seen_counts[n] = seen_counts.get(n, 0) + 1
        if total_counts[n] == 1:
            out.append(n)
        else:
            out.append(f"{n}_{seen_counts[n]}")
    return out


def extract_crosslinker(name: str) -> str:
    s = str(name)
    s = re.sub(r'_\d+$', '', s)
    parts = s.split("_")
    for key in CROSSLINKER_ORDER:
        if key in parts:
            return key
    return "OTHER"


def preprocess_signal(y, normalize_by_first_point=True):
    y = np.asarray(y, dtype=float)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    n_tail = max(5, int(len(y) * 0.05))
    baseline = np.median(y[-n_tail:])

    y_corr = y - baseline
    y_corr[y_corr < 0] = 0.0

    if normalize_by_first_point:
        first = y_corr[0] if y_corr[0] > 0 else np.max(y_corr)
        if first > 0:
            y_corr = y_corr / first

    return y_corr


def solve_ilt_ridge(y_processed, K, ridge_alpha):
    n_t2 = K.shape[1]
    Ireg = np.eye(n_t2)

    A_aug = np.vstack([K, ridge_alpha * Ireg])
    b_aug = np.concatenate([y_processed, np.zeros(n_t2)])

    res = lsq_linear(
        A_aug,
        b_aug,
        bounds=(0, np.inf),
        method="trf",
        lsmr_tol="auto",
        verbose=0
    )

    a = res.x
    fit = K @ a
    return a, fit, res


def calc_fit_r2(y, fit):
    ss_res = np.sum((y - fit) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    if ss_tot <= 0:
        return np.nan
    return 1.0 - ss_res / ss_tot


def classify_component_balance(short_f, mid_f, long_f):
    vals = {
        "short": short_f,
        "mid": mid_f,
        "long": long_f,
    }
    dominant = max(vals, key=vals.get)

    if dominant == "short":
        return "short-T2-dominant"
    elif dominant == "mid":
        return "mid-T2-dominant"
    elif dominant == "long":
        return "long-T2-dominant"
    return "component-unknown"


def classify_distribution_topology(n_peaks, width):
    if pd.isna(width):
        return "topology-unknown"

    if width >= BROAD_WIDTH_THRESHOLD:
        return "broad-distribution"

    if n_peaks <= 1:
        return "single-component"

    if n_peaks >= MULTI_PEAK_MIN:
        return "multi-component"

    return "two-component"


def classify_heterogeneity(entropy_norm, peak_gap):
    if pd.isna(entropy_norm):
        return "heterogeneity-unknown"

    if entropy_norm >= 0.70:
        return "high-heterogeneity"

    if entropy_norm >= 0.45:
        return "moderate-heterogeneity"

    if not pd.isna(peak_gap) and peak_gap >= 1.0:
        return "separated-components"

    return "low-heterogeneity"


def safe_peak_list(peaks):
    vals = []
    for p in peaks:
        if pd.isna(p):
            vals.append(np.nan)
        else:
            vals.append(float(p))
    while len(vals) < 4:
        vals.append(np.nan)
    return vals[:4]


def build_poly(x, z):
    verts = [(x[0], 0.0)]
    verts.extend(list(zip(x, z)))
    verts.append((x[-1], 0.0))
    return verts


# ============================================================
# 4. Plot helper functions
# ============================================================
def add_box_and_points(ax, df, group_col, value_col, ylabel, title,
                       use_log10=False, integer_y=False):
    groups = [g for g in CROSSLINKER_ORDER if g in df[group_col].unique()]
    if "OTHER" in df[group_col].unique():
        groups = groups + ["OTHER"]

    positions = np.arange(1, len(groups) + 1)

    plot_vals = []
    for g in groups:
        vals = df.loc[df[group_col] == g, value_col].astype(float).values
        if use_log10:
            vals = np.log10(np.clip(vals, 1e-12, None))
        plot_vals.append(vals)

    bp = ax.boxplot(
        plot_vals,
        positions=positions,
        widths=BOX_WIDTH,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black", linewidth=MEDIAN_LINEWIDTH),
        boxprops=dict(linewidth=BOX_LINEWIDTH),
        whiskerprops=dict(linewidth=BOX_LINEWIDTH),
        capprops=dict(linewidth=BOX_LINEWIDTH),
    )

    for patch, g in zip(bp["boxes"], groups):
        patch.set_facecolor(GROUP_COLORS.get(g, "#999999"))
        patch.set_alpha(BOX_FACE_ALPHA)
        patch.set_edgecolor(GROUP_COLORS.get(g, "#999999"))

    rng = np.random.default_rng(0)

    for pos, g, vals in zip(positions, groups, plot_vals):
        jitter = rng.uniform(-JITTER_WIDTH, JITTER_WIDTH, size=len(vals))

        ax.scatter(
            np.full(len(vals), pos) + jitter,
            vals,
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            color=GROUP_COLORS.get(g, "#999999"),
            edgecolors="black",
            linewidths=0.4,
            zorder=3
        )

        if len(vals) > 0:
            mean_val = np.mean(vals)
            ax.hlines(
                mean_val,
                pos - 0.20,
                pos + 0.20,
                colors="black",
                linewidth=2.0,
                zorder=4
            )

    ax.set_xticks(positions)
    ax.set_xticklabels(groups, fontsize=BOX_TICK_SIZE)
    ax.set_ylabel(ylabel, fontsize=BOX_LABEL_SIZE)
    ax.set_title(title, fontsize=BOX_TITLE_SIZE, pad=10)
    ax.tick_params(axis="y", labelsize=BOX_TICK_SIZE)
    ax.grid(axis="y", alpha=0.25, linestyle="--")

    if integer_y:
        ymin, ymax = ax.get_ylim()
        ymin_i = int(np.floor(ymin))
        ymax_i = int(np.ceil(ymax))
        ax.set_yticks(np.arange(ymin_i, ymax_i + 1, 1))


def plot_curtain_3d(meta_sorted, dist_sorted, T2_grid, sample_no,
                    group_boundaries, group_centers, zmax,
                    output_png, output_pdf=None):
    x = np.log10(T2_grid)

    verts = [build_poly(x, dist_sorted[i]) for i in range(len(dist_sorted))]
    facecolors = [
        CURTAIN_COLORS.get(c, CURTAIN_COLORS["OTHER"])
        for c in meta_sorted["Crosslinker"]
    ]

    fig = plt.figure(figsize=(CURTAIN_FIG_W, CURTAIN_FIG_H))
    ax = fig.add_subplot(111, projection="3d")

    poly = PolyCollection(
        verts,
        facecolors=facecolors,
        edgecolors=POLY_EDGE_COLOR,
        linewidths=POLY_EDGE_LW,
        alpha=FACE_ALPHA
    )
    ax.add_collection3d(poly, zs=sample_no, zdir="y")

    if SHOW_PROFILE_LINES:
        for i in range(len(dist_sorted)):
            ax.plot(
                x,
                np.full_like(x, sample_no[i], dtype=float),
                dist_sorted[i],
                color=PROFILE_LINE_COLOR,
                lw=PROFILE_LINE_LW,
                alpha=PROFILE_LINE_ALPHA,
                zorder=10
            )

    if SHOW_GROUP_SEPARATORS:
        for _, _, end_i in group_boundaries[:-1]:
            sep_y = end_i + 0.5
            ax.plot(
                [x.min(), XMAX_LOG10],
                [sep_y, sep_y],
                [0.0, 0.0],
                ls=GROUP_SEPARATOR_LS,
                lw=GROUP_SEPARATOR_LW,
                color=GROUP_SEPARATOR_COLOR
            )

    ax.set_xlabel(r"log10($T_2$ [ms])", fontsize=AXIS_LABEL_FONTSIZE, labelpad=X_LABELPAD)
    ax.set_ylabel("Sample No.", fontsize=AXIS_LABEL_FONTSIZE, labelpad=Y_LABELPAD)
    ax.set_zlabel("Amplitude", fontsize=AXIS_LABEL_FONTSIZE, labelpad=Z_LABELPAD)

    ax.set_xlim(x.min(), XMAX_LOG10)
    ax.set_ylim(1, len(sample_no) + 1)
    ax.set_zlim(0, zmax)

    ax.set_xticks(XTICKS_3)
    ax.set_xticklabels(XTICKLABELS_3)

    y_mid = int(round((1 + len(sample_no)) / 2))
    yticks = [1, y_mid, len(sample_no)]
    ax.set_yticks(yticks)
    ax.set_yticklabels([str(v) for v in yticks])

    zticks = [0.0, zmax / 2.0, zmax]
    ax.set_zticks(zticks)
    ax.set_zticklabels([f"{v:.1f}" for v in zticks])

    ax.tick_params(axis="x", labelsize=TICK_FONTSIZE_X)
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE_Y)
    ax.tick_params(axis="z", labelsize=TICK_FONTSIZE_Z)

    ax.view_init(elev=CURTAIN_ELEV, azim=CURTAIN_AZIM)

    try:
        ax.set_box_aspect((1.4, 1.8, 1.0))
    except Exception:
        pass

    if SHOW_GROUP_TEXT:
        for cl, center in group_centers:
            ax.text(
                GROUP_LABEL_X,
                center,
                0.02 * zmax,
                cl,
                color=CURTAIN_COLORS.get(cl, "black"),
                fontsize=GROUP_LABEL_FONTSIZE,
                ha="left",
                va="center"
            )

    plt.title("3D Stacked T2 Distributions", fontsize=TITLE_FONTSIZE)
    plt.tight_layout()
    plt.savefig(output_png, dpi=300, bbox_inches="tight")
    if output_pdf is not None:
        plt.savefig(output_pdf, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_feature_boxplots(feature_df, output_png, output_pdf=None):
    fig, axes = plt.subplots(1, 3, figsize=(BOX_FIG_W, BOX_FIG_H))
    axes = np.ravel(axes)

    add_box_and_points(
        axes[0],
        feature_df,
        "Crosslinker",
        "Weighted_logmean_T2_ms",
        ylabel=r"log10(weighted mean $T_2$ [ms])",
        title="Mobility",
        use_log10=True,
        integer_y=False
    )

    add_box_and_points(
        axes[1],
        feature_df,
        "Crosslinker",
        "Width_log10T2",
        ylabel=r"Distribution width in log10($T_2$)",
        title="Heterogeneity",
        use_log10=False,
        integer_y=False
    )

    add_box_and_points(
        axes[2],
        feature_df,
        "Crosslinker",
        "LongFraction",
        ylabel="Long-T2 fraction",
        title="Long fraction",
        use_log10=False,
        integer_y=False
    )

    plt.tight_layout()
    plt.savefig(output_png, dpi=BOX_DPI, bbox_inches="tight")
    if output_pdf is not None:
        plt.savefig(output_pdf, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ============================================================
# 5. Load data
# ============================================================
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"CSV file not found: {CSV_PATH}\n"
        "Colab左側のFilesで実際のファイル名を確認し、CSV_PATHを書き換えてください。"
    )

df = pd.read_csv(CSV_PATH)

raw_sample_names = df.iloc[:, 0].astype(str).fillna("Unknown").tolist()
sample_names = make_unique_names_with_index_for_duplicates(raw_sample_names)

time_ms = np.array([parse_float_ms(c) for c in df.columns[1:]], dtype=float)
Y_raw = df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").fillna(0.0).values

print("Loaded file:", CSV_PATH)
print("Number of samples:", len(sample_names))
print("Number of time points:", len(time_ms))
print("Time range (ms):", float(np.min(time_ms)), "to", float(np.max(time_ms)))


# ============================================================
# 6. Build ILT grid and kernel
# ============================================================
if USE_AUTO_T2_RANGE:
    T2_min_ms = max(0.1, np.min(time_ms) / 5.0)
    T2_max_ms = max(np.max(time_ms) * 20.0, 5000.0)
else:
    T2_min_ms = MANUAL_T2_MIN_MS
    T2_max_ms = MANUAL_T2_MAX_MS

T2_grid = np.logspace(np.log10(T2_min_ms), np.log10(T2_max_ms), N_T2)
K = np.exp(-np.outer(time_ms, 1.0 / T2_grid))

print("T2 grid range:", T2_min_ms, "to", T2_max_ms, "ms")
print("N_T2:", N_T2)
print("RIDGE_ALPHA:", RIDGE_ALPHA)


# ============================================================
# 7. Run ILT and extract all features
# ============================================================
dist_list = []
meta_records = []
feature_records = []

for idx, sample in enumerate(sample_names):

    y_proc = preprocess_signal(
        Y_raw[idx],
        normalize_by_first_point=NORMALIZE_BY_FIRST_POINT
    )

    a, fit, res = solve_ilt_ridge(y_proc, K, RIDGE_ALPHA)

    if np.sum(a) > 0:
        amp = a / np.sum(a)
    else:
        amp = a.copy()

    dist_list.append(amp.copy())

    logT2 = np.log10(T2_grid)

    weighted_logmean = np.sum(logT2 * amp)
    weighted_logmean_ms = 10 ** weighted_logmean

    width_log10T2 = np.sqrt(
        np.sum(amp * (logT2 - weighted_logmean) ** 2)
    )

    fit_r2 = calc_fit_r2(y_proc, fit)
    rmse = float(np.sqrt(np.mean((y_proc - fit) ** 2)))

    short_fraction = float(np.sum(amp[T2_grid < SHORT_T2_MAX_MS]))
    mid_fraction = float(
        np.sum(amp[(T2_grid >= SHORT_T2_MAX_MS) & (T2_grid < MID_T2_MAX_MS)])
    )
    long_fraction = float(np.sum(amp[T2_grid >= MID_T2_MAX_MS]))

    component_balance = classify_component_balance(
        short_fraction,
        mid_fraction,
        long_fraction
    )

    if np.max(amp) > 0:
        peak_indices, peak_props = find_peaks(
            amp,
            height=np.max(amp) * PEAK_HEIGHT_REL,
            distance=PEAK_DISTANCE_POINTS
        )
    else:
        peak_indices = np.array([], dtype=int)

    peak_T2_all = T2_grid[peak_indices]
    peak_amp_all = amp[peak_indices]

    if len(peak_T2_all) > 0:
        order = np.argsort(peak_amp_all)[::-1]
        peak_T2_ranked_by_amp = peak_T2_all[order]
        peak_amp_ranked = peak_amp_all[order]
    else:
        peak_T2_ranked_by_amp = np.array([])
        peak_amp_ranked = np.array([])

    peak_T2_list = safe_peak_list(peak_T2_ranked_by_amp)
    n_detected_peaks = int(len(peak_T2_all))

    if len(peak_T2_all) > 0:
        peak_T2_raw_max = float(np.max(peak_T2_all))
        strongest_peak_T2_ms = float(peak_T2_ranked_by_amp[0])
        strongest_peak_amp = float(peak_amp_ranked[0])
    else:
        peak_T2_raw_max = np.nan
        strongest_peak_T2_ms = np.nan
        strongest_peak_amp = np.nan

    if len(peak_T2_ranked_by_amp) >= 2:
        peak_gap_log10T2 = float(
            abs(
                np.log10(peak_T2_ranked_by_amp[0])
                - np.log10(peak_T2_ranked_by_amp[1])
            )
        )
    else:
        peak_gap_log10T2 = np.nan

    entropy = float(-np.sum(amp * np.log(amp + 1e-12)))
    entropy_norm = float(entropy / np.log(len(amp)))

    distribution_topology = classify_distribution_topology(
        n_detected_peaks,
        width_log10T2
    )

    heterogeneity_label = classify_heterogeneity(
        entropy_norm,
        peak_gap_log10T2
    )

    dynamics_vocabulary = "; ".join([
        component_balance,
        distribution_topology,
        heterogeneity_label
    ])

    crosslinker = extract_crosslinker(sample)

    meta_records.append({
        "Sample": sample,
        "Crosslinker": crosslinker
    })

    feature_records.append({
        "Original_name": raw_sample_names[idx],
        "Sample": sample,
        "Crosslinker": crosslinker,

        # code1-compatible ILT columns
        "Weighted_logmean_T2_ms": weighted_logmean_ms,
        "Width_log10T2": width_log10T2,
        "Fit_R2": fit_r2,
        "Ridge_alpha": RIDGE_ALPHA,
        "N_detected_peaks": n_detected_peaks,
        "Peak_T2_ms_raw_max": peak_T2_raw_max,

        "Peak1_T2_ms": peak_T2_list[0],
        "Peak2_T2_ms": peak_T2_list[1],
        "Peak3_T2_ms": peak_T2_list[2],
        "Peak4_T2_ms": peak_T2_list[3],

        # additional ILT descriptors
        "Strongest_peak_T2_ms": strongest_peak_T2_ms,
        "Strongest_peak_amplitude": strongest_peak_amp,
        "RMSE": rmse,

        # component fraction
        "ShortFraction": short_fraction,
        "MidFraction": mid_fraction,
        "LongFraction": long_fraction,
        "Component_balance": component_balance,

        # distribution topology
        "Distribution_topology": distribution_topology,

        # heterogeneity
        "Entropy": entropy,
        "Entropy_norm": entropy_norm,
        "PeakGap_log10T2": peak_gap_log10T2,
        "Heterogeneity_label": heterogeneity_label,

        # language-ready descriptor
        "Dynamics_vocabulary": dynamics_vocabulary,

        # reproducibility parameters
        "Short_T2_max_ms": SHORT_T2_MAX_MS,
        "Mid_T2_max_ms": MID_T2_MAX_MS,
        "Peak_height_relative_threshold": PEAK_HEIGHT_REL,
        "Peak_distance_points": PEAK_DISTANCE_POINTS,
        "Broad_width_threshold": BROAD_WIDTH_THRESHOLD,
        "N_T2_grid": N_T2,
        "T2_grid_min_ms": T2_min_ms,
        "T2_grid_max_ms": T2_max_ms,
        "Normalize_by_first_point": NORMALIZE_BY_FIRST_POINT,
    })


# ============================================================
# 8. Create dataframes
# ============================================================
meta_df = pd.DataFrame(meta_records)
feature_df = pd.DataFrame(feature_records)
dist_array = np.array(dist_list)

# ============================================================
# 9. Sort by crosslinker for figures
# ============================================================
ordered_idx = []

for cl in CROSSLINKER_ORDER:
    idxs = meta_df[meta_df["Crosslinker"] == cl].index.tolist()
    ordered_idx.extend(idxs)

other_idxs = [i for i in range(len(meta_df)) if i not in ordered_idx]
ordered_idx.extend(other_idxs)

meta_sorted = meta_df.loc[ordered_idx].reset_index(drop=True)
dist_sorted = dist_array[ordered_idx]
feature_df_sorted = feature_df.loc[ordered_idx].reset_index(drop=True)

sample_no = np.arange(1, len(meta_sorted) + 1)

group_boundaries = []
group_centers = []

start = 0
for cl in CROSSLINKER_ORDER:
    count = int(np.sum(meta_sorted["Crosslinker"] == cl))
    if count == 0:
        continue
    end = start + count
    group_boundaries.append((cl, start + 1, end))
    group_centers.append((cl, (start + 1 + end) / 2))
    start = end

z_data_max = float(np.nanmax(dist_sorted))
zmax = z_data_max * 1.05 if ZMAX_CURTAIN is None else ZMAX_CURTAIN

print("Data z max:", z_data_max)
print("Plot z max:", zmax)


# ============================================================
# 10. Save CSV files
# ============================================================
summary_csv = os.path.join(OUTDIR, "ilt_summary_ridge_improved.csv")
summary_xlsx = os.path.join(OUTDIR, "ilt_summary_ridge_improved.xlsx")

feature_df_sorted.to_csv(summary_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
    feature_df_sorted.to_excel(writer, index=False, sheet_name="ilt_summary")

# Optional: save ILT distribution matrix
dist_csv = os.path.join(OUTDIR, "ilt_distributions_ridge_improved.csv")

dist_df = pd.DataFrame(
    dist_sorted,
    columns=[f"T2_{v:.6g}_ms" for v in T2_grid]
)
dist_df.insert(0, "Crosslinker", meta_sorted["Crosslinker"].values)
dist_df.insert(0, "Sample", meta_sorted["Sample"].values)
dist_df.to_csv(dist_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 11. Generate figures
# ============================================================
ilt_png = os.path.join(OUTDIR, "ILT_3D_stacked_crosslinker_no_legend.png")
ilt_pdf = os.path.join(OUTDIR, "ILT_3D_stacked_crosslinker_no_legend.pdf")

box_png = os.path.join(OUTDIR, "Feature_group_comparison_crosslinker_1x3.png")
box_pdf = os.path.join(OUTDIR, "Feature_group_comparison_crosslinker_1x3.pdf")

plot_curtain_3d(
    meta_sorted=meta_sorted,
    dist_sorted=dist_sorted,
    T2_grid=T2_grid,
    sample_no=sample_no,
    group_boundaries=group_boundaries,
    group_centers=group_centers,
    zmax=zmax,
    output_png=ilt_png,
    output_pdf=ilt_pdf
)

plot_feature_boxplots(
    feature_df=feature_df_sorted,
    output_png=box_png,
    output_pdf=box_pdf
)


# ============================================================
# 12. ZIP export
# ============================================================
zip_path = os.path.join(OUTDIR, "AddD2O_CPMG_ILT_summary_figures.zip")

files_to_zip = [
    summary_csv,
    summary_xlsx,
    dist_csv,
    ilt_png,
    ilt_pdf,
    box_png,
    box_pdf,
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in files_to_zip:
        if os.path.exists(fp):
            zf.write(fp, arcname=os.path.basename(fp))

print("\nSaved files:")
for fp in files_to_zip:
    print(" -", fp)

print("\nZIP:")
print(" -", zip_path)

print("\nPreview of ilt_summary_ridge_improved.csv:")
display(feature_df_sorted.head())

print("\nQC summary:")
print("Number of samples:", len(feature_df_sorted))
print("Mean Fit_R2:", feature_df_sorted["Fit_R2"].mean())
print("Rows with missing Peak1_T2_ms:", feature_df_sorted["Peak1_T2_ms"].isna().sum())
print("Rows with missing PeakGap_log10T2:", feature_df_sorted["PeakGap_log10T2"].isna().sum())

# ============================================================
# 13. Download ZIP in Colab
# ============================================================
from google.colab import files
files.download(zip_path)